# Singapore Banks Regime-Aware Fuzzy-Gated LSTM Project

AI6124 Group 4. Authors: ZHOU YIQING and MANIAR KUSH HEMAL. Add student IDs before submission.

This notebook is a reproducible entry point. No final performance claims are included. See `docs/RESEARCH_PROTOCOL.md`.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path.cwd().resolve()
if not (PROJECT / "src/uob_regime").is_dir():
    PROJECT = PROJECT.parent
if not (PROJECT / "src/uob_regime").is_dir():
    raise RuntimeError("Open from the project root or notebooks directory")
sys.path.insert(0, str(PROJECT / "src"))

from uob_regime.config import load_config
BANK = "uob"  # Choose uob, dbs or ocbc; each bank has its own data/results.
CONFIGS = {"uob": "default.json", "dbs": "dbs.json", "ocbc": "ocbc.json"}
config = load_config(PROJECT / "configs" / CONFIGS[BANK])
print("Project:", PROJECT)
print("Models:", config["models"])

## Data acquisition

Run the CLI download command once before continuing. Inspect raw metadata and corporate actions independently. No data is automatically downloaded here.

In [ ]:
from uob_regime.experiment import prepare
fold_table = prepare(config)
fold_table

## Historical smoke experiment

Set the switch below deliberately. Five epochs check the pipeline, not final performance.

In [ ]:
from uob_regime.experiment import run_experiments

RUN_HISTORICAL_SMOKE = False
if RUN_HISTORICAL_SMOKE:
    smoke_config = {**config, "models": ["persistence", "lstm", "fuzzy_moe"], "seeds": [11], "epochs": 5}
    run_path = run_experiments(smoke_config, years=[2020])
    print(run_path)

## Inspect completed historical results

Use the exact returned run directory. Do not inspect final holdout results while tuning.

In [ ]:
import pandas as pd

RUN_DIRECTORY = None
if RUN_DIRECTORY is not None:
    run_directory = Path(RUN_DIRECTORY)
    if not run_directory.name.startswith("historical_"):
        raise ValueError("Select historical results during development")
    display(pd.read_csv(run_directory / "metrics.csv"))
    display(pd.read_csv(run_directory / "seed_summary.csv"))

## Literature review and method

TODO: Add verified primary literature on TSK/ANFIS, LSTM, expert mixtures and fuzzy recurrent forecasting. Explain the adapted fuzzy-feature baseline and 27-rule TSK gate. Distinguish our extension from reference code.

## Results and interpretation

The complete three-bank development comparison is exported to `reports/multi_bank/REPORT.md`. It covers nine methods on six annual folds (2020-2025), with three seeds for each learned model. Annual-fold returns are separate portfolios, not a compounded six-year investment. See `summary.csv`, `annual_metrics.csv`, `annual_variability.csv`, `seed_ensemble_metrics.csv` and `cost_sensitivity.csv` for the reproducible tables.

UOB development results show that zero-return forecasting has the best RMSE; the fuzzy-feature LSTM has the strongest learned-model directional and trading results. The TSK gate is close to equal weighting, so clear expert specialization is unsupported. DBS and OCBC provide additional checks using the same pipeline. Their conclusions must be drawn from the saved comparison report, not assumed from UOB. The banks share the STI market context and are correlated.

In [ ]:
from IPython.display import display, Markdown, Image
REPORTS = PROJECT / "reports/multi_bank"
display(Markdown((REPORTS / "REPORT.md").read_text()))
display(pd.read_csv(REPORTS / "summary.csv"))
display(Image(filename=str(REPORTS / "comparison.png")))

## Final holdout

Freeze reviewed settings with the CLI `freeze` command, then use `run-holdout --frozen-config ...`. This notebook intentionally contains no automatic held-out training cell.

## Limitations and reproducibility

Discuss vendor adjustments, forecast/execution horizon mismatch, limited data, cost assumptions and evidence of expert specialization. Record data hashes, environment details and run IDs. See the teammate handoff document for supplementary baselines.